In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HAPPENN)
This notebook curates the **HAPPENN** hemolysis dataset from a FASTA-like file that is not strictly compliant with the standard FASTA format. The original file contains a leading line index (a number and a space) at the beginning of each line, so we first rewrite it into a clean temporary FASTA file. We then parse sequences, infer binary labels from record identifiers, run duplicate consistency checks, and export the final curated dataset and metadata.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HAPPENN
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Preprocesses a non-standard FASTA export**:
  - removes leading `"<number><space>"` prefixes from each line,
  - writes a temporary standard FASTA file and parses it with a FASTA reader,
  - deletes the temporary file after parsing.
- **Infers hemolysis labels from FASTA record IDs**:
  - `label = 1` if the record `id` contains `"hemolytic"`,
  - `label = 0` if the record `id` contains `"non-hemolytic"`,
  - otherwise the label remains `NA` (kept as missing).
- **Keeps a standardized schema**: `(sequence, label)`.
- **Checks duplicates by sequence**:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged and exported as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports outputs**:
  - `processed_hemolytic_dataset.csv` (deduplicated curated dataset),
  - `detected_error_sequences.csv` (conflicting-label duplicates),
  - `metadata.json`.

In [2]:
name_source = "HAPPENN"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
# Initially, the file is not in a standard FASTA format, as each line contains a leading number and a space
# The file is rewritten into a temporary file, which is then read as a standard FASTA file

temp_fasta = "archivo.fasta"

with open(f"{PATH_INPUT}/{name_source}/41598_2020_67701_MOESM1_ESM.fasta") as fin, open(temp_fasta, "w") as fout:
    for line in fin:
        fout.write(re.sub(r"^\d+\s+", "", line))

df = read_fasta_doc(temp_fasta)
os.remove(temp_fasta)

In [4]:
df["label"] = pd.NA 

df.loc[df["id"].str.contains("hemolytic", case=False, na=False), "label"] = 1
df.loc[df["id"].str.contains("non-hemolytic", case=False, na=False), "label"] = 0

df = df[["sequence", "label"]]
df.shape

(3738, 2)

- Checking duplicates

In [5]:
df["sequence"].unique().shape

(3551,)

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(3532, 2)

In [8]:
df_errors.shape

(19, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2020,
 'last update date': 'No information',
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta;pdf',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Experimentally validated, "Characteristic threshold (IC50, MIC, etc.)"',
 'repository or server': 'https://research.timmons.eu/happenn',
 'publication': 'https://www.nature.com/articles/s41598-020-67701-3#Sec19',
 'number_of_raw_sequences': 3738,
 'number_of_sequences_retained': 3532,
 'number_of_positive_sequences': 1456,
 'number_of_negative_sequences': 2076,
 'number_of_erroneous_sequences': 19,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)